<a href="https://colab.research.google.com/github/VISHWAJA-028/ML_Multimodality/blob/main/pcos_fusion_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
"""
pcos_fusion.py  -  export / import / infer for the multimodal PCOS project

  build_bundle(...)   -> packs structured model + preprocessing + ultrasound CNN
                         + fusion settings into ONE file:  pcos_multimodal.pkl
  PCOSPredictor(path) -> loads that file and predicts from tabular data,
                         an ultrasound image, or both (late fusion)

Everything the notebooks do to the raw data (cleaning, feature engineering,
one-hot encoding, column-name cleaning) is re-implemented below so that a raw
row from the Excel sheet can be fed straight in.
"""
import json
import os
import pickle
import re
import tempfile
import warnings

import joblib
import numpy as np
import pandas as pd

TARGET = "PCOS (Y/N)"
DROP_COLS = ["Source_Dataset", "Sl. No", "Patient File No."]
ONEHOT_COLS = ["Blood Group", "Cycle(R/I)", "BMI_Category"]

BG_MAP = {"11": "A+", "12": "A-", "13": "B+", "14": "B-",
          "15": "O+", "16": "O-", "17": "AB+", "18": "AB-"}
CYCLE_MAP = {"2": "R", "4": "I", "R": "R", "I": "I"}


# --------------------------------------------------------------------------
# Structured-data preprocessing (mirrors ML_Final notebook cells 4, 5, 10, 11)
# --------------------------------------------------------------------------
def _fix_double_decimal(x):
    s = str(x)
    if s.count(".") > 1:
        first = s.index(".")
        s = s[: first + 1] + s[first + 1:].replace(".", "")
    return s


def _bmi_category(bmi):
    if bmi < 18.5:
        return "Underweight"
    elif bmi < 25:
        return "Normal"
    elif bmi < 30:
        return "Overweight"
    return "Obese"


def _clean_col_names(df):
    new_cols = []
    for col in df.columns:
        c = re.sub(r"[^A-Za-z0-9_]", "_", col)
        c = re.sub(r"_{2,}", "_", c).strip("_")
        new_cols.append(c)
    seen, final = {}, []
    for col in new_cols:
        base, n = col, 1
        while col in seen:
            col = f"{base}_{n}"
            n += 1
        seen[col] = True
        final.append(col)
    df = df.copy()
    df.columns = final
    return df


def _clean(raw):
    df = raw.copy()
    df = df.drop(columns=[TARGET] + DROP_COLS, errors="ignore")

    df["Blood Group"] = df["Blood Group"].astype(str).str.strip().replace(BG_MAP)
    df["Cycle(R/I)"] = df["Cycle(R/I)"].astype(str).str.strip().map(CYCLE_MAP)
    df["AMH(ng/mL)"] = pd.to_numeric(df["AMH(ng/mL)"], errors="coerce")
    df["II beta-HCG(mIU/mL)"] = pd.to_numeric(
        df["II beta-HCG(mIU/mL)"].apply(_fix_double_decimal), errors="coerce")

    for c in [c for c in df.columns if "(Y/N)" in c]:
        df[c] = (df[c].astype(str).str.strip().str.upper()
                 .map({"Y": 1, "N": 0, "1": 1, "0": 0}))
    return df


def _derive_categories(cleaned):
    """Category lists seen at training time (get_dummies sorts them)."""
    bmi_cat = cleaned["BMI"].apply(_bmi_category)
    return {
        "Blood Group": sorted(cleaned["Blood Group"].dropna().unique()),
        "Cycle(R/I)": sorted(cleaned["Cycle(R/I)"].dropna().unique()),
        "BMI_Category": sorted(bmi_cat.dropna().unique()),
    }


def _engineer(cleaned, categories):
    df = cleaned.copy()
    df["Follicle_Total"] = df["Follicle No. (L)"] + df["Follicle No. (R)"]
    df["Avg_F_Size_Mean"] = df[["Avg. F size (L) (mm)",
                                "Avg. F size (R) (mm)"]].mean(axis=1)
    df["BMI_Category"] = df["BMI"].apply(_bmi_category)
    # Fixed categories => every dummy column exists even for a single row
    for col in ONEHOT_COLS:
        df[col] = pd.Categorical(df[col], categories=categories[col])
    df = pd.get_dummies(df, columns=ONEHOT_COLS, dummy_na=True)
    return _clean_col_names(df)


def preprocess_structured(raw, bundle):
    """Raw Excel-style rows -> DataFrame aligned to the training columns."""
    fe = _engineer(_clean(raw), bundle["categories"])
    fe = fe.drop(columns=["PCOS_Y_N"], errors="ignore")
    return fe.reindex(columns=bundle["columns"]).astype(float)


# --------------------------------------------------------------------------
# EXPORT
# --------------------------------------------------------------------------
def build_bundle(art_dir, excel_path, out_path, struct_auc,
                 image_auc=None, weights=None,
                 keras_name="PCOS_best_image_model.keras",
                 info_name="PCOS_image_model_info.json"):
    """
    art_dir     folder holding the files the two notebooks already saved
                (structured_*.joblib, PCOS_best_image_model.keras, ...info.json)
    excel_path  pocs_final.xlsx  (used only to recover the training categories
                and to verify the rebuilt features match the trained columns)
    struct_auc  AUC of the structured model on its test set (from ML_Final)
    image_auc   defaults to roc_auc stored in the image model's info json
    weights     optional {"structured": w1, "ultrasound": w2}; default is
                proportional to each model's AUC
    """
    model = joblib.load(os.path.join(art_dir, "structured_model.joblib"))
    scaler = joblib.load(os.path.join(art_dir, "structured_scaler.joblib"))
    imputer = joblib.load(os.path.join(art_dir, "structured_imputer.joblib"))
    columns = joblib.load(os.path.join(art_dir, "structured_columns.joblib"))

    with open(os.path.join(art_dir, info_name)) as f:
        info = json.load(f)
    with open(os.path.join(art_dir, keras_name), "rb") as f:
        keras_bytes = f.read()

    raw = pd.read_excel(excel_path)
    categories = _derive_categories(_clean(raw))
    rebuilt = _engineer(_clean(raw), categories).drop(columns=["PCOS_Y_N"], errors="ignore")
    if list(rebuilt.columns) != list(columns):
        missing = set(columns) - set(rebuilt.columns)
        extra = set(rebuilt.columns) - set(columns)
        raise ValueError(f"Feature mismatch.\n  missing: {missing}\n  extra: {extra}")

    image_auc = image_auc if image_auc is not None else info["roc_auc"]
    if weights is None:
        tot = struct_auc + image_auc
        weights = {"structured": struct_auc / tot, "ultrasound": image_auc / tot}

    import sklearn
    versions = {"sklearn": sklearn.__version__}
    for mod in ("xgboost", "lightgbm", "tensorflow"):
        try:
            versions[mod] = __import__(mod).__version__
        except Exception:
            pass

    bundle = {
        "structured": {"model": model, "imputer": imputer, "scaler": scaler},
        "columns": list(columns),
        "categories": categories,
        "image": {
            "keras_bytes": keras_bytes,
            "img_size": tuple(info["image_size"]),
            "class_mapping": info["class_mapping"],
            "pcos_class": "infected",     # <- change if 'noninfected' is your PCOS-positive class
        },
        "fusion": {"weights": weights, "threshold": 0.5},
        "metrics": {"structured_auc": struct_auc, "image_auc": image_auc},
        "versions": versions,
    }
    with open(out_path, "wb") as f:
        pickle.dump(bundle, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"Saved {out_path}  ({os.path.getsize(out_path) / 1e6:.1f} MB)")
    print("Fusion weights:", weights)
    return bundle


# --------------------------------------------------------------------------
# IMPORT + INFERENCE
# --------------------------------------------------------------------------
class PCOSPredictor:
    def __init__(self, path):
        with open(path, "rb") as f:          # only unpickle files you created yourself
            self.b = pickle.load(f)

        import sklearn
        if self.b["versions"].get("sklearn") != sklearn.__version__:
            warnings.warn(f"scikit-learn version differs (saved "
                          f"{self.b['versions'].get('sklearn')}, now {sklearn.__version__})")

        self.model = self.b["structured"]["model"]
        self.imputer = self.b["structured"]["imputer"]
        self.scaler = self.b["structured"]["scaler"]
        self.weights = self.b["fusion"]["weights"]
        self.threshold = self.b["fusion"]["threshold"]
        self._cnn = None                      # loaded lazily (TensorFlow is slow to import)

    # ---- structured -------------------------------------------------------
    def predict_structured(self, row):
        """row: dict or DataFrame with the raw Excel column names."""
        raw = pd.DataFrame([row]) if isinstance(row, dict) else row
        X = preprocess_structured(raw, self.b)
        X = pd.DataFrame(self.imputer.transform(X), columns=X.columns)
        X = pd.DataFrame(self.scaler.transform(X), columns=X.columns)
        return float(self.model.predict_proba(X)[:, 1][0])      # P(PCOS)

    # ---- ultrasound -------------------------------------------------------
    def _load_cnn(self):
        if self._cnn is None:
            from tensorflow import keras
            with tempfile.TemporaryDirectory() as d:
                p = os.path.join(d, "image_model.keras")
                with open(p, "wb") as f:
                    f.write(self.b["image"]["keras_bytes"])
                self._cnn = keras.models.load_model(p, compile=False)
        return self._cnn

    def predict_image(self, image_path, n_tta=5):
        from tensorflow.keras.utils import load_img, img_to_array
        cnn = self._load_cnn()
        x = img_to_array(load_img(image_path, target_size=self.b["image"]["img_size"]))[None]
        # The notebook builds the model with augmentation forced on (training=True),
        # so it is still active at inference -> average several passes.
        p = float(np.mean([cnn(x, training=False).numpy().ravel()[0] for _ in range(n_tta)]))
        pos = self.b["image"]["pcos_class"]
        return p if self.b["image"]["class_mapping"][pos] == 1 else 1.0 - p   # P(PCOS)

    # ---- fusion -----------------------------------------------------------
    def predict(self, tabular=None, image_path=None, n_tta=5):
        parts = {}
        if tabular is not None:
            parts["structured"] = self.predict_structured(tabular)
        if image_path is not None:
            parts["ultrasound"] = self.predict_image(image_path, n_tta)
        if not parts:
            raise ValueError("Provide tabular data, an ultrasound image, or both.")

        w = {k: self.weights[k] for k in parts}
        p = sum(w[k] * parts[k] for k in parts) / sum(w.values())
        return {
            "p_structured": parts.get("structured"),
            "p_ultrasound": parts.get("ultrasound"),
            "p_pcos": p,
            "prediction": "PCOS" if p >= self.threshold else "Non-PCOS",
            "used": list(parts),
        }